In [1]:
import pandas as pd

In [2]:
# Get all sheet names
file = 'profiles/pithos_output/processing_scripts/HEC-PITHOS_high_techcost.xlsx'

xls = pd.ExcelFile(file)
sheet_names = xls.sheet_names
# Read all sheets into a DataFrame list
df_list = []
for sheet in sheet_names:
    print(sheet)
    _df = xls.parse(sheet)
    # infer types of the column names
    _df.columns = _df.columns.astype(str)
    df_list.append(_df)
# Combine all DataFrames into one
df = pd.concat(df_list, ignore_index=True)

Cost-Capital
Cost-FOM
Cost-VOM
Cost-Fuel
Cost-Carbon
Cap-Gen_New
Cap-Gen_Tot
Cap-TrL_New
Cap-TrL-Tot
Emissions
Dispatch-AB_to_NL
Dispatch-NS_to_SK
Dispatch-TrL_Flow


In [4]:

df.columns = df.columns.str.lower()

In [5]:
classes = df["variable"].apply(lambda x: x.split("|")[0])

In [15]:
transmission_df = df[classes == 'Transmission flow'].copy()
transmission_df = transmission_df.dropna(axis=1, how='all')
transmission_df["variable"] = transmission_df["variable"].apply(lambda x: '|'.join(x.split("|")[1:]))


In [16]:
from profiles.pithos_output import utils

# replace to with ''
transmission_df['variable'] = transmission_df['variable'].str.replace('to ', '')
transmission_df["variable"] = transmission_df.variable.apply(lambda x: x.split(".")[0])
transmission_df = transmission_df.melt(id_vars=['variable', 'region', 'hour', 'scenario', 'model'], var_name='time',
                                           value_name='value')
transmission_df['time'] = pd.to_datetime(transmission_df['time'].astype(str) + '-01-01') + pd.to_timedelta(transmission_df['hour'], unit='h')



In [17]:
transmission_df.head()

,variable,region,hour,scenario,model,time,value
0,BC,AB,1.0,high_techcost,HEC-PITHOS,2035-01-01 01:00:00,0.0
1,BC,AB,2.0,high_techcost,HEC-PITHOS,2035-01-01 02:00:00,0.0
2,BC,AB,3.0,high_techcost,HEC-PITHOS,2035-01-01 03:00:00,0.0
3,BC,AB,4.0,high_techcost,HEC-PITHOS,2035-01-01 04:00:00,0.0
4,BC,AB,5.0,high_techcost,HEC-PITHOS,2035-01-01 05:00:00,0.0


In [18]:
transmission_df['time'] = transmission_df['time'] - pd.Timedelta(hours=1)
transmission_df['period'] = transmission_df['time'].dt.year
# make period an int
transmission_df['period'] = transmission_df['period'].astype(int)

transmission_df.drop(columns=['time'], inplace=True)

# drop all the 0 values
# transmission_df = transmission_df[transmission_df.value != 0]
transmission_df['value'] = transmission_df['value'] * -1

# aggregate df values by region, variable, time, hour
transmission_df = transmission_df.groupby(["region", "variable", "period"]).sum().reset_index()
# rename from and variable based on utils.province_short
transmission_df["region"] = transmission_df.region.map(utils.province_short).fillna(transmission_df['region'])
transmission_df["variable"] = transmission_df.variable.map(utils.province_short).fillna(transmission_df['variable'])

# drop rows where region == variable
transmission_df = transmission_df[transmission_df.region != transmission_df.variable]



In [19]:
transmission_df.head()

,region,variable,period,hour,scenario,model,value
0,AB,BC,2021,38373180.0,high_techcosthigh_techcosthigh_techcosthigh_te...,HEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PI...,0.00
1,AB,BC,2025,38373180.0,high_techcosthigh_techcosthigh_techcosthigh_te...,HEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PI...,-1015285.06
2,AB,BC,2030,38373180.0,high_techcosthigh_techcosthigh_techcosthigh_te...,HEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PI...,-2473567.93
3,AB,BC,2035,38373180.0,high_techcosthigh_techcosthigh_techcosthigh_te...,HEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PI...,-1457277.13
4,AB,BC,2040,38373180.0,high_techcosthigh_techcosthigh_techcosthigh_te...,HEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PI...,-818618.44


In [20]:
# create a dataframe where the region and variable are swapped and the value is -1* the value
transmission_df_swap = transmission_df.copy()
transmission_df_swap["region"] = transmission_df["variable"]
transmission_df_swap["variable"] = transmission_df["region"]
transmission_df_swap["value"] = -1 * transmission_df["value"]

transmission_df = pd.concat([transmission_df, transmission_df_swap], ignore_index=True)
transmission_df["region"] = transmission_df.region.apply(lambda x: x.split(".")[0])
# rename region entries based on utils.province_short
transmission_df['region'] = transmission_df['region'].map(utils.province_short).fillna(transmission_df['region'])
dim_names = []
for index, row in transmission_df.iterrows():
    if row['value'] < 0:
        dim_names.append(f"Exports to {row['variable']}")
    else:
        dim_names.append(f"Imports from {row['variable']}")
transmission_df['variable'] = dim_names


In [21]:
transmission_df.head()

,region,variable,period,hour,scenario,model,value
0,AB,Imports from BC,2021,38373180.0,high_techcosthigh_techcosthigh_techcosthigh_te...,HEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PI...,0.00
1,AB,Exports to BC,2025,38373180.0,high_techcosthigh_techcosthigh_techcosthigh_te...,HEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PI...,-1015285.06
2,AB,Exports to BC,2030,38373180.0,high_techcosthigh_techcosthigh_techcosthigh_te...,HEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PI...,-2473567.93
3,AB,Exports to BC,2035,38373180.0,high_techcosthigh_techcosthigh_techcosthigh_te...,HEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PI...,-1457277.13
4,AB,Exports to BC,2040,38373180.0,high_techcosthigh_techcosthigh_techcosthigh_te...,HEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PITHOSHEC-PI...,-818618.44
